In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec, GridSpecFromSubplotSpec
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

font_path      = 'font-folder/Roboto-Light.ttf'
font_path_bold = 'font-folder/Roboto-Bold.ttf'
font_prop         = fm.FontProperties(fname=font_path,      size=14)
font_prop_small   = fm.FontProperties(fname=font_path,      size=9)
font_prop_numbers = fm.FontProperties(fname=font_path,      size=9)
font_prop_title   = fm.FontProperties(fname=font_path_bold, size=14)

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
tsfm_models = [
    'Chronos-2', 'Lag-Llama',
    'Moirai-2',  'TabPFN-TS',
    'TimeGPT', 'TimesFM'
]
tsfm_display = ['Chronos-2', 'Lag-Llama', 'Moirai-2', 'TabPFN-TS', 'TimeGPT', 'TimesFM']

model_palettes = {
    'Chronos-2':          '#4ECED4',
    'Moirai-2':           '#967ECD',
    'Lag-Llama':          '#3E977E',
    'TabPFN-TS':          '#CE5858',
    'TimeGPT':            '#1F77B4',
    'TimesFM':            '#F2B134',
    'FluSight-ensemble':  'teal',
    'CovidHub-ensemble':  'teal',
}

cat_colors = {
    'large_decrease': '#2166AC',  # blu scuro — declino ripido
    'decrease':        '#92C5DE',  # azzurro — declino lieve
    'stable':          '#B0B0B0',  # grigio — stabile
    'increase':        '#F4A582',  # arancio — crescita lieve
    'large_increase':  '#D6604D',  # rosso — crescita ripida
}

cat_labels_display = {
    'large_decrease': 'Large decrease',
    'decrease':        'Decrease',
    'stable':          'Stable',
    'increase':        'Increase',
    'large_increase':  'Large increase',
}
SCORES_FILES = {
    'FluSight': '../model-evaluation/forecast_scores/forecast_scores_wis_ae_coverage_2024-2025_log_FluSight.csv',
    'CovidHub': '../model-evaluation/forecast_scores/forecast_scores_wis_ae_coverage_2024-2025_log_CovidHub.csv',
}
TARGET_FILES_TH = {
    'FluSight': '../target-data/FluSight/surveillance_weekly.csv',
    'CovidHub': '../target-data/CovidHub/surveillance_weekly.csv',
}
OUTPUT_DIR = '../model-evaluation/figures/plot_trend_category/'

BASELINE = {'FluSight': 'FluSight-baseline', 'CovidHub': 'CovidHub-baseline'}
ENSEMBLE = {'FluSight': 'FluSight-ensemble', 'CovidHub': 'CovidHub-ensemble'}

US_NATIONAL_LOCATION = 'US'

LOG_TICKS = [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 2.5, 3.0]

display_names = {
    'FluSight-ensemble': 'hub-ensemble',
    'CovidHub-ensemble': 'hub-ensemble',
    'Other models':      'Other models',
}

In [ ]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

# ============================================================
# COMPUTE RAW COUNT DIFF AND RATE DIFF PER LOCATION
# ============================================================
def compute_diff_and_rate(grp):
    grp = grp.sort_values('truth_date').copy()
    grp['obs_lag'] = grp['obs'].shift(1)
    grp['diff'] = grp['obs'] - grp['obs_lag']
    grp['rate'] = grp['obs'] / grp['population'] * 1e5
    grp['rate_lag'] = grp['rate'].shift(1)
    grp['rate_diff'] = grp['rate'] - grp['rate_lag']
    return grp


# ============================================================
# ASSIGN SEASON (unchanged)
# ============================================================
def assign_season(date):
    if date.month >= 10:
        return f"{date.year}/{date.year+1}"
    else:
        return f"{date.year-1}/{date.year}"

# ============================================================
# CDC FIXED-THRESHOLD CLASSIFIER (given thresholds, applies to any hub)
# ============================================================
def classify_trend_fixed(df_surv, rate_thresholds, target_seasons=None, count_floor=10):
    df_surv = df_surv.copy()
    mask = df_surv['season'].isin(target_seasons) if target_seasons is not None else pd.Series(True, index=df_surv.index)

    rate_diff = df_surv.loc[mask, 'rate_diff']
    count_diff = df_surv.loc[mask, 'diff']

    cat = pd.Series(index=rate_diff.index, dtype=object)
    cat[rate_diff <= rate_thresholds['large_decrease']] = 'large_decrease'
    cat[(rate_diff > rate_thresholds['large_decrease']) & (rate_diff < rate_thresholds['decrease'])] = 'decrease'
    cat[(rate_diff >= rate_thresholds['decrease']) & (rate_diff <= rate_thresholds['increase'])] = 'stable'
    cat[(rate_diff > rate_thresholds['increase']) & (rate_diff < rate_thresholds['large_increase'])] = 'increase'
    cat[rate_diff >= rate_thresholds['large_increase']] = 'large_increase'

    cat[count_diff.abs() < count_floor] = 'stable'
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    df_surv['trend_cat'] = np.nan
    df_surv.loc[mask, 'trend_cat'] = cat
    df_surv['trend_cat'] = pd.Categorical(df_surv['trend_cat'], categories=cat_labels_ordered, ordered=True)
    return df_surv

TARGET_SEASONS = ['2024/2025', '2025/2026']
EXCLUDED_SEASONS_COVID = ['2020/2021', '2021/2022']  # add seasons to exclude if needed (e.g. very early pandemic period)

def derive_thresholds_from_reference(df, target_seasons, excluded_seasons=None, national_location='US'):
    excluded_seasons = excluded_seasons or []
    ref = df[
        (df['location'] == national_location) &
        ~df['season'].isin(excluded_seasons)
    ]
    p5, p25, p75, p95 = np.percentile(ref['rate_diff'].dropna(), [5, 25, 75, 95])
    return {
        'large_decrease': p5, 'decrease': p25,
        'increase': p75, 'large_increase': p95
    }, len(ref)


# ============================================================
# LOAD SURVEILLANCE — all locations
# ============================================================
def load_surveillance_with_bins(challenge):
    df_surv = pd.read_csv(TARGET_FILES_TH[challenge])
    df_surv['truth_date'] = pd.to_datetime(df_surv['truth_date'])
    df_surv = (
        df_surv
        .drop_duplicates(['truth_date', 'location'])
        .sort_values(['location', 'truth_date'])
        .rename(columns={'value': 'obs'})
        [['truth_date', 'location', 'obs']]
        .copy()
    )

    # LOAD POPULATION (needed to convert obs -> rate per 100k)
    url_locations = "https://raw.githubusercontent.com/cdcepi/FluSight-forecast-hub/main/auxiliary-data/locations.csv"
    df_pop = pd.read_csv(url_locations)
    # expect columns like: abbreviation, location, location_name, population

    df_pop = df_pop.rename(columns={'location': 'location'})[['location', 'population']]

    df_surv = df_surv.merge(df_pop, on='location', how='left')
    print("Missing population for:", df_surv[df_surv['population'].isna()]['location'].unique())
    df_surv = df_surv.groupby('location', group_keys=False).apply(compute_diff_and_rate)
    df_surv = df_surv.dropna(subset=['diff', 'rate_diff'])
    df_surv['season'] = df_surv['truth_date'].apply(assign_season)

    
    flu_thresholds, n_ref = derive_thresholds_from_reference(df_surv, TARGET_SEASONS, excluded_seasons=EXCLUDED_SEASONS_COVID)
    print(challenge, flu_thresholds)
    df_surv_flu = classify_trend_fixed(df_surv, flu_thresholds, target_seasons=TARGET_SEASONS)
    return df_surv_flu


# --- Scores / Cleveland ---
def load_scores_with_bins(challenge, df_surv):
    baseline_challenge = BASELINE[challenge]
    df = pd.read_csv(SCORES_FILES[challenge])
    df['reference_date'] = pd.to_datetime(df['reference_date'])

    baseline_df = df[df['model'] == baseline_challenge][
        ['reference_date', 'target', 'target_end_date', 'horizon', 'location', 'wis', 'ae_median']
    ].rename(columns={'wis': 'wis_baseline', 'ae_median': 'ae_median_baseline'})

    df = df.merge(baseline_df,
                  on=['reference_date', 'target', 'target_end_date', 'horizon', 'location'],
                  how='left')
    df['wis_relative']       = df['wis']       / df['wis_baseline']
    df['ae_median_relative'] = df['ae_median'] / df['ae_median_baseline']
    df = df.drop(columns=['wis_baseline', 'ae_median_baseline'])

    df['model_name'] = df['model'].copy()

    ref_loc_pairs = df[['reference_date', 'location']].drop_duplicates()
    def find_bin(row):
        loc_surv = df_surv[df_surv['location'] == row['location']]
        if loc_surv.empty:
            return np.nan
        idx = (loc_surv['truth_date'] - row['reference_date']).abs().idxmin()
        return loc_surv.loc[idx, 'trend_cat']
    ref_loc_pairs['trend_cat'] = ref_loc_pairs.apply(find_bin, axis=1)
    df = df.merge(ref_loc_pairs[['reference_date', 'location', 'trend_cat']],
                  on=['reference_date', 'location'], how='left')
    return df

def compute_medians(df, metric, challenge):
    baseline_challenge = BASELINE[challenge]
    ensemble_name      = ENSEMBLE[challenge]
    exclude = set(tsfm_display) | {ensemble_name, baseline_challenge}

    tsfm_med = (
        df[df['model_name'].isin(tsfm_display) & df['trend_cat'].notna()]
        .groupby(['model_name', 'trend_cat'])[metric].median().reset_index()
    )
    ens_med = (
        df[(df['model_name'] == ensemble_name) & df['trend_cat'].notna()]
        .groupby('trend_cat')[metric].median().reset_index()
    )
    other_med = (
        df[~df['model_name'].isin(exclude) &
           ~df['model'].isin([baseline_challenge]) &
           df['trend_cat'].notna()]
        .groupby(['model_name', 'trend_cat'])[metric].median().reset_index()
        .groupby('trend_cat')[metric].mean().reset_index()
    )
    return tsfm_med, ens_med, other_med

print('Functions defined.')

In [ ]:
# ============================================================
# LOAD ALL DATA
# ============================================================
print('Loading FluSight surveillance...')
df_surv_flu = load_surveillance_with_bins('FluSight')
df_surv_flu_all = df_surv_flu.copy()
df_surv_flu = df_surv_flu[(df_surv_flu['truth_date'] >= '2024-11-21') & (df_surv_flu['truth_date'] <= '2025-06-21')]
print('Loading FluSight scores...')
df_scores_flu = load_scores_with_bins('FluSight', df_surv_flu)
df_scores_flu = df_scores_flu[(df_scores_flu['target_end_date'] >= '2024-11-21') & (df_scores_flu['target_end_date'] <= '2025-06-21')]
print('Loading CovidHub surveillance...')
df_surv_covid   = load_surveillance_with_bins('CovidHub')
df_surv_covid = df_surv_covid[(df_surv_covid['truth_date'] >= '2024-11-21') & (df_surv_covid['truth_date'] <= '2025-06-21')]
print('Loading CovidHub scores...')
df_scores_covid = load_scores_with_bins('CovidHub', df_surv_covid)
df_scores_covid = df_scores_covid[(df_scores_covid['target_end_date'] >= '2024-11-21') & (df_scores_covid['target_end_date'] <= '2025-06-21')]

data = {
    'FluSight': df_scores_flu,
    'CovidHub': df_scores_covid,
}
surv = {
    'FluSight': df_surv_flu,
    'CovidHub': df_surv_covid,
}
print('Done.')

In [ ]:
df_surv_flu_all = df_surv_flu_all[df_surv_flu_all['season'] >= '2024/2025']
df_surv_flu_all.head()

In [ ]:
df_surv_flu_all.to_csv('../model-evaluation/df_surv_flu_trend_cat_CDC_adapted.csv', index=False)

In [ ]:
# ============================================================
# PREPARE TREND PANEL DATA  (stacked bar + US curve)
# ============================================================
def prepare_trend_data(challenge):
    df_surv = surv[challenge]
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    # States only (exclude US national from the bar count)
    df_states = df_surv[df_surv['location'] != US_NATIONAL_LOCATION].dropna(subset=['trend_cat'])
    counts = (
        df_states
        .groupby(['truth_date', 'trend_cat'])
        .size()
        .reset_index(name='n_states')
    )
    pivot = (
        counts
        .pivot(index='truth_date', columns='trend_cat', values='n_states')
        .reindex(columns=cat_labels_ordered)
        .fillna(0)
    )

    # US national curve
    df_us = df_surv[df_surv['location'] == US_NATIONAL_LOCATION].sort_values('truth_date')

    return pivot, df_us

pivot_flu,   df_us_flu   = prepare_trend_data('FluSight')
pivot_covid, df_us_covid = prepare_trend_data('CovidHub')
print('Trend data ready.')

In [ ]:
# ============================================================
# PLOTTING FUNCTIONS
# ============================================================

def plot_trend_panel(ax_bar, ax_line, pivot, df_us, title, df_surv, add_legend=False):
    """Draw stacked-bar (ax_bar) + US national line (ax_line) for one challenge."""
    dates  = pivot.index
    bottom = np.zeros(len(dates))
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    for col in cat_labels_ordered:
        vals = pivot[col].values
        ax_bar.bar(
            dates, vals, bottom=bottom,
            color=cat_colors[col],
            width=pd.Timedelta(days=6),
            linewidth=0,
            label=col.replace('\n', ' ')
        )
        bottom = bottom + vals

    ax_bar.set_ylabel('Number of \nU.S. states', fontproperties=font_prop)
    ax_bar.tick_params(axis='x', labelbottom=False)
    ax_bar.set_title(title, fontproperties=font_prop_title, loc='left', pad=8)
    for spine in ['top', 'right']:
        ax_bar.spines[spine].set_visible(False)
    ax_bar.spines['bottom'].set_linewidth(0.4)
    ax_bar.spines['left'].set_linewidth(0.4)
    ax_bar.grid(axis='y', alpha=0.15, linewidth=0.4)
    for lbl in ax_bar.get_yticklabels():
        lbl.set_fontproperties(font_prop)

    if add_legend:
        legend_handles = [
            Patch(facecolor=cat_colors[b], label=b.replace('\n', ' '))
            for b in cat_labels_ordered
        ]
        ax_bar.legend(
            handles=legend_handles,
            loc='upper left',
            bbox_to_anchor=(1.01, 1),
            bbox_transform=ax_bar.transAxes,
            frameon=False,
            prop=font_prop,
            ncol=1
        )

    # State curves on secondary y-axis (light grey)
    ax_line2 = ax_line.twinx()
    df_states_all = df_surv[df_surv['location'] != US_NATIONAL_LOCATION].sort_values('truth_date')
    for loc, grp in df_states_all.groupby('location'):
        ax_line2.plot(
            grp['truth_date'], grp['obs'],
            color='#CCCCCC', linewidth=0.4, alpha=0.5, zorder=1
        )
    ax_line2.set_ylabel('')
    # US national line
    ax_line.plot(df_us['truth_date'], df_us['obs'], marker = 'o',
                 color='#2C3E50', linewidth=1.4)
    ax_line.set_xlabel(' ', fontproperties=font_prop)
    for spine in ['top', 'right']:
        ax_line.spines[spine].set_visible(False)
    ax_line.spines['bottom'].set_linewidth(0.4)
    ax_line.spines['left'].set_linewidth(0.4)
    ax_line.grid(axis='y', alpha=0.15, linewidth=0.4)
    for lbl in ax_line.get_xticklabels():
        lbl.set_fontproperties(font_prop)
        lbl.set_rotation(30)
        lbl.set_ha('right')
    for lbl in ax_line.get_yticklabels():
        lbl.set_fontproperties(font_prop)
    
    ax_line.set_ylabel('Hospitalizations', fontproperties=font_prop, color='#2C3E50')
    ax_line.tick_params(axis='y', colors='#2C3E50')
    for lbl in ax_line.get_yticklabels():
        lbl.set_fontproperties(font_prop)
        lbl.set_color('#2C3E50')

    ax_line2.set_ylabel('', fontproperties=font_prop, color='#AAAAAA')
    ax_line2.tick_params(axis='y', colors='#AAAAAA', right=True, labelright=True)
    for lbl in ax_line2.get_yticklabels():
        lbl.set_fontproperties(font_prop)
        lbl.set_color('#AAAAAA')
    for spine in ax_line2.spines.values():
        spine.set_visible(False)
    # annotate US national
    ax_line.annotate(
        'US national',
        xy=(df_us['truth_date'].iloc[-1], df_us['obs'].iloc[-1]),
        xytext=(8, 0),
        textcoords='offset points',
        va='center',
        fontproperties=font_prop,
        color='#2C3E50'
    )
    # annotate US states
    ax_line2.annotate(
        'US states',
        xy=(df_states_all['truth_date'].iloc[-1], df_states_all.groupby('location')['obs'].last().iloc[-1]),
        xytext=(8, 0),
        textcoords='offset points',
        va='center',
        fontproperties=font_prop,
        color='#AAAAAA'
    )


def build_cleveland_dataframe(challenge, metric):
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    df = data[challenge]
    tsfm_med, ens_med, other_med = compute_medians(df, metric, challenge)
    ensemble_name = ENSEMBLE[challenge]
    rows = []
    for b in cat_labels_ordered:
        val = ens_med.loc[ens_med['trend_cat'] == b, metric]
        if len(val):
            rows.append({'model': ensemble_name, 'trend_cat': b, 'value': val.values[0]})
    for model in tsfm_display:
        sub = tsfm_med[tsfm_med['model_name'] == model]
        for b in cat_labels_ordered:
            val = sub.loc[sub['trend_cat'] == b, metric]
            if len(val):
                rows.append({'model': model, 'trend_cat': b, 'value': val.values[0]})
    for b in cat_labels_ordered:
        val = other_med.loc[other_med['trend_cat'] == b, metric]
        if len(val):
            rows.append({'model': 'Other models', 'trend_cat': b, 'value': val.values[0]})
    return pd.DataFrame(rows)


def plot_cleveland_panel(ax, challenge, metric, title):
    """Draw a single Cleveland dot-plot panel."""
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    df_plot = build_cleveland_dataframe(challenge, metric)
    ensemble_name = ENSEMBLE[challenge]

    local_order = [
        'Chronos-2', 'Lag-Llama', 'Moirai-2', 'TabPFN-TS',
        'TimeGPT', 'TimesFM', 'Other models', ensemble_name,
    ]
    y_positions = np.arange(len(local_order))[::-1]

    for y, model in zip(y_positions, local_order):
        sub = df_plot[df_plot['model'] == model]
        for b in cat_labels_ordered:
            val = sub.loc[sub['trend_cat'] == b, 'value']
            if len(val):
                ax.scatter(val.values[0], y, s=50,
                           color=cat_colors[b], edgecolor='none', zorder=3)

    ax.axvline(1.0, linestyle='--', linewidth=0.9, color='#888888', alpha=0.8, zorder=2)
    ax.set_yticks(y_positions)
    ax.set_yticklabels(
        [display_names.get(m, m) for m in local_order],
        fontproperties=font_prop
    )
    ax.set_xscale('log')
    ax.xaxis.set_major_locator(ticker.FixedLocator(LOG_TICKS))
    ax.xaxis.set_minor_locator(ticker.NullLocator())
    ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{x:g}'))
    ax.xaxis.offsetText.set_visible(False)
    for lbl in ax.get_xticklabels():
        lbl.set_fontproperties(font_prop)
    ax.set_xlim(0.42, 3.2)
    ax.set_title(title, fontproperties=font_prop_title, loc='left', x=0.0, pad=8)
    xlabel = ('Median WIS ratio (model / baseline)' if metric == 'wis_relative'
              else 'Median AE ratio (model / baseline)')
    ax.set_xlabel(xlabel, fontproperties=font_prop)
    ax.grid(axis='x', alpha=0.15, linewidth=0.4)
    for spine in ['top', 'right', 'left']:
        ax.spines[spine].set_visible(False)
    ax.spines['bottom'].set_linewidth(0.5)
    ax.tick_params(axis='y', length=0)

print('Plot functions defined.')

In [ ]:
# ============================================================
# BUILD COMBINED FIGURE
#
#   Row 0  |  A – FluSight trend (bar + line)  |  B – CovidHub trend (bar + line)
#   Row 1  |  C – FluSight WIS  |  D – FluSight AE  |  E – CovidHub WIS  |  F – CovidHub AE
#            (row 1 spans all 4 columns via a nested 1×4 GridSpec)
#
# ============================================================
cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
fig = plt.figure(figsize=(20, 10), dpi=300)

outer_gs = GridSpec(
    2, 1,
    figure=fig,
    height_ratios=[2.0, 1.6],
    hspace=0.38
)

# ── Row 0: Trend panels A and B  ─────────────────────────
trend_gs = GridSpecFromSubplotSpec(
    1, 2,
    subplot_spec=outer_gs[0],
    wspace=0.28
)

for col_idx, (challenge, panel_label, pivot, df_us) in enumerate([
    ('FluSight', 'A. Flu epidemic dynamic', pivot_flu,   df_us_flu),
    ('CovidHub', 'B. COVID-19 epidemic dynamic', pivot_covid, df_us_covid),
]):
    inner_gs = GridSpecFromSubplotSpec(
        2, 1,
        subplot_spec=trend_gs[0, col_idx],
        height_ratios=[1.4, 1],
        hspace=0.08
    )
    ax_bar  = fig.add_subplot(inner_gs[0])
    ax_line = fig.add_subplot(inner_gs[1], sharex=ax_bar)

    # Add legend only on panel A (right side, outside)
    add_leg = (col_idx == 0)
    plot_trend_panel(ax_bar, ax_line, pivot, df_us, panel_label, surv[challenge], add_legend=add_leg)

# ── Row 1: Cleveland panels C D E F ─────────────────────
cleveland_gs = GridSpecFromSubplotSpec(
    1, 4,
    subplot_spec=outer_gs[1],
    wspace=0.35
)

cleveland_specs = [
    # (col, challenge, metric, label)
    (0, 'FluSight', 'wis_relative',      'C. WIS'),
    (1, 'FluSight', 'ae_median_relative', 'D. AE'),
    (2, 'CovidHub', 'wis_relative',      'E. WIS'),
    (3, 'CovidHub', 'ae_median_relative', 'F. AE'),
]

for col, challenge, metric, label in cleveland_specs:
    ax = fig.add_subplot(cleveland_gs[0, col])
    plot_cleveland_panel(ax, challenge, metric, label)
    # Hide y-tick labels on panels D, E, F to avoid repetition
    if col > 0:
        ax.set_yticklabels([])
        ax.set_ylabel('')

# ── Shared legend for Cleveland panels (bottom centre) ────────────────────
cleveland_legend_handles = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=cat_colors[b],
           markersize=10, label=b.replace('\n', ' '))
    for b in cat_labels_ordered
] + [
    Line2D([0], [0], linestyle='--', color='#888888', label='Baseline (1.0)')
]

fig.legend(
    handles=cleveland_legend_handles,
    loc='lower center',
    ncol=5,
    frameon=False,
    prop=font_prop,
    bbox_to_anchor=(0.5, 0.005)
)

# ── Save ──────────────────────────────────────────────────────────────────
out_base = f'{OUTPUT_DIR}trend_combined_panel_2024-2025'
fig.savefig(out_base + '.png', dpi=300, bbox_inches='tight')
fig.savefig(out_base + '.svg', bbox_inches='tight')
plt.show()
print('Figure saved.')

In [ ]:
# ============================================================
# TOP 3 tra gli 8 modelli di confronto (6 TSFM + hub-ensemble + "Other models")
# per ogni epidemic phase, usando la stessa aggregazione del Cleveland plot
# ============================================================

def top3_per_phase_cleveland(challenge, metric, top_n=3):
    cat_labels_ordered = ['large_decrease', 'decrease', 'stable', 'increase', 'large_increase']
    df_plot = build_cleveland_dataframe(challenge, metric)  # stessa aggregazione del plot: TSFM + ensemble + "Other models"

    results = []
    for phase in cat_labels_ordered:
        sub = df_plot[df_plot['trend_cat'] == phase].sort_values('value')
        top = sub.head(top_n).copy()
        top['rank'] = range(1, len(top) + 1)
        top['challenge'] = challenge
        top['metric'] = metric
        top['phase'] = phase
        results.append(top[['challenge', 'metric', 'phase', 'rank', 'model', 'value']]
                        .rename(columns={'value': 'median_value'}))

    return pd.concat(results, ignore_index=True)


# --- Esegui per tutte le combinazioni challenge x metric ---
all_top3 = []
for challenge in ['FluSight', 'CovidHub']:
    for metric in ['wis_relative', 'ae_median_relative']:
        all_top3.append(top3_per_phase_cleveland(challenge, metric))

df_top3 = pd.concat(all_top3, ignore_index=True)
pd.set_option('display.max_rows', None)
df_top3